In [ ]:
# ============================================================
# 05_machine_learning.ipynb — PRÉDICTION DU RISQUE DE RETARD
# ============================================================

import sys
sys.path.append('..')
from src.utils import *
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, roc_auc_score)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 1 — CHARGEMENT")
print("="*60 + "\n")
# -------------------------------------------------------

data = pd.read_csv('../data/processed/logistique_ml_ready.csv')
print(f"✓ Table chargée : {data.shape}")

y = data['est_en_retard']
X = data.drop(columns=['est_en_retard'])

print(f"✓ Features : {X.shape[1]}")
print(f"✓ Taux de retard : {y.mean()*100:.2f}%")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 2 — TRAIN / TEST SPLIT")
print("="*60 + "\n")
# -------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✓ Train : {X_train.shape[0]:,} lignes ({y_train.mean()*100:.2f}% retard)")
print(f"✓ Test  : {X_test.shape[0]:,} lignes ({y_test.mean()*100:.2f}% retard)")


# -------------------------------------------------------
print("\n" + "="*60)
print("   ÉTAPE 3 — NORMALISATION (pour Logistic Regression)")
print("="*60 + "\n")
# -------------------------------------------------------

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("✓ StandardScaler fit sur train uniquement")


# -------------------------------------------------------
print("\n" + "="*60)
print("   MODÈLE 1 — LOGISTIC REGRESSION")
print("="*60 + "\n")
# -------------------------------------------------------

lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_prob_lr)

print(f"Accuracy : {acc_lr*100:.1f}%")
print(f"AUC-ROC  : {auc_lr:.3f}")
print(classification_report(y_test, y_pred_lr, target_names=['À l\'heure','En retard']))


# -------------------------------------------------------
print("\n" + "="*60)
print("   MODÈLE 2 — RANDOM FOREST")
print("="*60 + "\n")
# -------------------------------------------------------

rf = RandomForestClassifier(n_estimators=150, max_depth=12,
                             class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_prob_rf)

print(f"Accuracy : {acc_rf*100:.1f}%")
print(f"AUC-ROC  : {auc_rf:.3f}")
print(classification_report(y_test, y_pred_rf, target_names=['À l\'heure','En retard']))

cm = confusion_matrix(y_test, y_pred_rf)
print(f"\n--- Matrice de confusion ---")
print(f"  VN={cm[0,0]:>6}  FP={cm[0,1]:>6}")
print(f"  FN={cm[1,0]:>6}  VP={cm[1,1]:>6}")


# -------------------------------------------------------
print("\n" + "="*60)
print("   COMPARAISON")
print("="*60 + "\n")
# -------------------------------------------------------

print(f"  {'Modèle':<20} | {'Accuracy':>9} | {'AUC':>8}")
print(f"  {'-'*42}")
print(f"  {'Logistic Regression':<20} | {acc_lr*100:>8.1f}% | {auc_lr:>8.3f}")
print(f"  {'Random Forest':<20} | {acc_rf*100:>8.1f}% | {auc_rf:>8.3f}")


# -------------------------------------------------------
print("\n" + "="*60)
print("   FEATURE IMPORTANCE — Validation des hypothèses EDA")
print("="*60 + "\n")
# -------------------------------------------------------

importances = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(importances.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
top10 = importances.head(10)
ax.barh(top10['feature'], top10['importance'], color='#1D9E75')
ax.invert_yaxis()
ax.set_title('Random Forest — Feature Importance (Top 10)')
plt.tight_layout()
plt.savefig('../reports/figures/ml_feature_importance_retard.png', dpi=150)
plt.show()


# -------------------------------------------------------
print("\n" + "="*60)
print("   EXPORT RÉSULTATS")
print("="*60 + "\n")
# -------------------------------------------------------

resultats = pd.DataFrame({
    'modele': ['Logistic Regression', 'Random Forest'],
    'accuracy': [acc_lr, acc_rf],
    'auc_roc': [auc_lr, auc_rf]
})
resultats.to_csv('../data/exports/ml_resultats_retard.csv', index=False)
importances.to_csv('../data/exports/feature_importance_retard.csv', index=False)

print("✓ Résultats exportés")
print(f"\n→ Meilleur modèle : {'Random Forest' if auc_rf > auc_lr else 'Logistic Regression'}")